In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_D.xpt"
csv_file = DATA_DIR / "SMQFAM_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 SMQFAM_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD430"
]


if len(df) != 10348:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 5:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "SMQFAM_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: SMQFAM_D structure = "
    "10,348 rows x 5 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: SMQFAM_D unexpectedly contains "
        "tiny XPORT values. "
        "Do NOT automatically convert them to zero."
    )


print(
    "PASS: SMQFAM_D contains no tiny-value artifacts."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected source zero(s) found. "
        "Review against CDC before proceeding."
    )


print(
    "PASS: Original SMQFAM_D contains no numeric zeros."
)


# ============================================================
# 6. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# We perform this check BEFORE converting anything
# to Int64.
#
# If a genuine fractional field exists, STOP rather than
# destroying its source-decoded decimal representation.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


if fractional_columns:

    print(
        "Fractional variables found:"
    )

    print(
        fractional_columns
    )

    raise ValueError(
        "STOP: Genuine decimal variables found. "
        "Do NOT convert those fields to Int64."
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:

    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31127:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC CHECK - SMD410
# ============================================================

#
# Does anyone smoke in home?
#
# CDC:
#
# 1 Yes     = 1,759
# 2 No      = 8,477
# 7 Refused = 3
# 9 DK      = 0
# Missing   = 109
#

EXPECTED_SMD410 = {
    1: 1759,
    2: 8477,
    7: 3
}


actual_smd410 = (
    df["SMD410"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smd410 = {
    int(k): int(v)
    for k, v
    in actual_smd410.items()
}


if actual_smd410 != EXPECTED_SMD410:

    raise ValueError(
        "SMD410 frequency mismatch.\n"
        f"Expected: {EXPECTED_SMD410}\n"
        f"Actual:   {actual_smd410}"
    )


if int(
    df["SMD410"]
    .isna()
    .sum()
) != 109:

    raise ValueError(
        "SMD410 missing count differs."
    )


print(
    "\nPASS: SMD410 matches CDC."
)


# ============================================================
# 9. CDC CHECK - SMD415
# ============================================================

#
# Total number of smokers in home.
#
# CDC:
#
# 1-2 = 1,621 total
# 3   = 138
# Missing = 8,589
#
# Actual XPT detail:
#
# 1 = 1,117
# 2 =   504
# 3 =   138
#

EXPECTED_SMD415 = {
    1: 1117,
    2: 504,
    3: 138
}


actual_smd415 = (
    df["SMD415"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smd415 = {
    int(k): int(v)
    for k, v
    in actual_smd415.items()
}


if actual_smd415 != EXPECTED_SMD415:

    raise ValueError(
        "SMD415 frequencies differ.\n"
        f"Expected: {EXPECTED_SMD415}\n"
        f"Actual:   {actual_smd415}"
    )


if int(
    df["SMD415"]
    .isna()
    .sum()
) != 8589:

    raise ValueError(
        "SMD415 missing count differs."
    )


if (
    actual_smd415[1]
    +
    actual_smd415[2]
) != 1621:

    raise ValueError(
        "SMD415 CDC range total differs."
    )


print(
    "PASS: SMD415 matches CDC/source."
)


# ============================================================
# 10. CDC CHECK - SMD415A
# ============================================================

#
# Total number of cigarette smokers in home.
#
# CDC:
#
# 1-2 = 1,545 total
# 3   = 135
# Missing = 8,668
#
# Actual source:
#
# 1 = 1,059
# 2 =   486
# 3 =   135
#

EXPECTED_SMD415A = {
    1: 1059,
    2: 486,
    3: 135
}


actual_smd415a = (
    df["SMD415A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smd415a = {
    int(k): int(v)
    for k, v
    in actual_smd415a.items()
}


if actual_smd415a != EXPECTED_SMD415A:

    raise ValueError(
        "SMD415A frequencies differ.\n"
        f"Expected: {EXPECTED_SMD415A}\n"
        f"Actual:   {actual_smd415a}"
    )


if int(
    df["SMD415A"]
    .isna()
    .sum()
) != 8668:

    raise ValueError(
        "SMD415A missing count differs."
    )


if (
    actual_smd415a[1]
    +
    actual_smd415a[2]
) != 1545:

    raise ValueError(
        "SMD415A CDC range total differs."
    )


print(
    "PASS: SMD415A matches CDC/source."
)


# ============================================================
# 11. CDC CHECK - SMD430
# ============================================================

#
# Total cigarettes smoked inside home per day.
#
# CDC public release:
#
# 1-39 = 1,492
# 40   = 172   ("40 or more")
# Missing = 8,684
#
# There are NO released zeros in the attached XPT.
#

s = df["SMD430"]


if not (
    int(
        s.between(
            1,
            39
        ).sum()
    ) == 1492

    and

    int(
        s.eq(40)
        .sum()
    ) == 172

    and

    int(
        s.isna()
        .sum()
    ) == 8684
):

    raise ValueError(
        "SMD430 differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected SMD430 zero values."
    )


print(
    "PASS: SMD430 matches CDC."
)

print(
    "      1-39 cigarettes = 1,492"
)

print(
    "      40 or more      = 172"
)

print(
    "      Missing         = 8,684"
)


# ============================================================
# 12. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# SMQFAM_D requires NO numeric corrections.
#
# No:
#
# - tiny -> zero correction
# - rounding
# - recalculation
# - decimal conversion
#
# expected_df therefore equals the original XPT exactly.
#

expected_df = source_df.copy()


# ============================================================
# 13. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# All five fields were verified to contain only
# exact whole-number values.
#
# Therefore Int64 is safe.
#
# Examples:
#
# SEQN
# 31127.0 -> 31127
#
# SMD410
# 1.0 -> 1
#
# SMD415
# 3.0 -> 3
#
# SMD430
# 40.0 -> 40
#
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and SMQFAM_D fields "
    "converted to clean integer formatting."
)


# ============================================================
# 14. PRE-EXPORT EXACT VALUE FIDELITY
# ============================================================

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: No SMQFAM_D numeric values changed."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# float_format
# round()
# recalculation
#
# Missing SAS values become blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 16. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 18. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- XPT -> CSV NUMERIC FIDELITY ---"
)


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 5 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 19. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 20. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(
        reader
    )


if len(raw_rows) != 10348:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 21. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# All five SMQFAM_D fields are proven
# whole-number numeric fields.
#
# Expected representation:
#
# missing -> ""
#
# otherwise:
# str(int(source_value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "SMQFAM_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number representation."
)


# ============================================================
# 22. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 23. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 24. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 25. FINAL CDC CHECK AFTER EXPORT
# ============================================================

print(
    "\n--- FINAL CSV CDC CHECK ---"
)


final_smd410 = (
    csv_df["SMD410"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_smd410 = {
    int(k): int(v)
    for k, v
    in final_smd410.items()
}


if final_smd410 != EXPECTED_SMD410:

    raise ValueError(
        "SMD410 changed after export."
    )


final_smd415 = (
    csv_df["SMD415"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_smd415 = {
    int(k): int(v)
    for k, v
    in final_smd415.items()
}


if final_smd415 != EXPECTED_SMD415:

    raise ValueError(
        "SMD415 changed after export."
    )


final_smd415a = (
    csv_df["SMD415A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_smd415a = {
    int(k): int(v)
    for k, v
    in final_smd415a.items()
}


if final_smd415a != EXPECTED_SMD415A:

    raise ValueError(
        "SMD415A changed after export."
    )


if not (
    int(
        csv_df["SMD430"]
        .between(
            1,
            39
        )
        .sum()
    ) == 1492

    and

    int(
        csv_df["SMD430"]
        .eq(40)
        .sum()
    ) == 172

    and

    int(
        csv_df["SMD430"]
        .isna()
        .sum()
    ) == 8684
):

    raise ValueError(
        "SMD430 changed after export."
    )


print(
    "PASS: Final CSV distributions still match CDC."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SMQFAM_D CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny artifacts in original XPT:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    zero_total
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code/count formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "SMQFAM_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 SMQFAM_D XPT -> CSV
Rows: 10,348
Columns: 5
PASS: SMQFAM_D structure = 10,348 rows x 5 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: SMQFAM_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Original SMQFAM_D contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
PASS: Genuine decimal variables = 0.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31127.0
Maximum: 41474.0
PASS: SEQN validated.

PASS: SMD410 matches CDC.
PASS: SMD415 matches CDC/source.
PASS: SMD415A matches CDC/source.
PASS: SMD430 matches CDC.
      1-39 cigarettes = 1,492
      40 or more      = 172
      Missing         = 8,684

PASS: SEQN and SMQFAM_D fields converted to clean integer formatting.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: No SMQFAM_D numeric values changed.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\2005_2006\Questionnnair\SMQFAM_D_corrected.csv

